In [4]:
import random
import numpy as np

In [5]:
import argparse
import numpy as np
from PIL import Image

## DB scan class
class DB_scan:
    def __init__(self, r, min_samples, name_distance):
        self.r = r
        self.min_samples = min_samples
        self.name_distance = name_distance
## Store data in the class
    def fit(self, X):
        self.X = X
        self.n = len(X)
        self.labels = [-1] * self.n  ## For recognising points as unvisited and also for recognising points with -1 as noise
## Defining the three types of distances
    def distance(self, a, b):
        if self.name_distance == "Euclidiean":
            return np.sqrt(np.sum((a - b) ** 2))
        if self.name_distance == "Manhattan":
            return np.sum(np.abs(a - b))
        if self.name_distance == "Chebyshev":
            return np.max(np.abs(a - b))
## Defining clustering
    def cluster(self):
        cluster_id = 0

        for i in range(self.n):
            if self.labels[i] != -1: ## For recognising points as unvisited
                continue
## Applying the first condition of the DB Scan with respect to parameter r
            neighbors = []
            for j in range(self.n):
                if self.distance(self.X[i], self.X[j]) <= self.r:
                    neighbors.append(j)## Nighbours is the list of indexes  in X of points in clusters not points themselves
## Applying second condition w.r.t parameter min_points or samples
            if len(neighbors) < self.min_samples:
                self.labels[i] = -1
                continue

            self.labels[i] = cluster_id
            k = 0

            while k < len(neighbors):
                p = neighbors[k]

                if self.labels[p] == -1:
                    self.labels[p] = cluster_id
## Testing same two conditions for points  whose indices are in neighbours.
                new_neighbors = []
                for q in range(self.n):
                    if self.distance(self.X[p], self.X[q]) <= self.r:
                        new_neighbors.append(q)

                if len(new_neighbors) >= self.min_samples:
                    for q in new_neighbors:
                        if q not in neighbors:
                            neighbors.append(q)

                k += 1

            cluster_id += 1



### Testing on iris

from sklearn.datasets import load_iris

iris = load_iris()
X = iris.data        # shape: (150, 4)
y = iris.target      # true labels (not used by clustering)


dbscan = DB_scan(r=1.5, min_samples=5
                 , name_distance="Euclidiean")
dbscan.fit(X)
dbscan.cluster()

labels_dbscan = np.array(dbscan.labels)

print("DBSCAN cluster labels:")
print(labels_dbscan)





DBSCAN cluster labels:
[0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
 1 1]


In [6]:
# command line interface
def main():
    parser = argparse.ArgumentParser(description="DBSCAN image segmentation")
    parser.add_argument("--image", required=True, help="Path to input image")
    parser.add_argument("--eps", type=float, required=True)
    parser.add_argument("--min_samples", type=int, required=True)
    parser.add_argument("--distance", required=True,
                        choices=["Euclidiean", "Manhattan", "Chebyshev"])
    parser.add_argument("--output", default="dbscan_output.png")

    args = parser.parse_args()

    img = Image.open(args.image)
    img_np = np.array(img)
    h, w, _ = img_np.shape

    pixels = img_np.reshape(-1, 3).astype(float)

    model = DB_scan(args.eps, args.min_samples, args.distance)
    model.fit(pixels)
    model.cluster()

    labels = np.array(model.labels)
    unique_labels = set(labels)

    segmented = np.zeros_like(pixels)
    for lbl in unique_labels:
        if lbl == -1:
            continue
        mean_color = pixels[labels == lbl].mean(axis=0)
        segmented[labels == lbl] = mean_color

    segmented = segmented.reshape(h, w, 3).astype(np.uint8)
    Image.fromarray(segmented).save(args.output)


if __name__ == "__main__":
    main()

    ## example out put:  python dbscan.py --image input.png --eps 10 --min_samples 5 --distance Euclidiean --output result.png


usage: colab_kernel_launcher.py [-h] --image IMAGE --eps EPS --min_samples
                                MIN_SAMPLES --distance
                                {Euclidiean,Manhattan,Chebyshev}
                                [--output OUTPUT]
colab_kernel_launcher.py: error: the following arguments are required: --image, --eps, --min_samples, --distance
ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.



Traceback (most recent call last):
  File "/usr/lib/python3.12/argparse.py", line 1943, in _parse_known_args2
    namespace, args = self._parse_known_args(args, namespace, intermixed)
                      ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/argparse.py", line 2230, in _parse_known_args
    raise ArgumentError(None, _('the following arguments are required: %s') %
argparse.ArgumentError: the following arguments are required: --image, --eps, --min_samples, --distance

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py", line 3553, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
  File "/tmp/ipython-input-873201805.py", line 38, in <cell line: 0>
    main()
  File "/tmp/ipython-input-873201805.py", line 11, in main
    args = parser.parse_args()
           ^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/pyt

TypeError: object of type 'NoneType' has no len()